### 1. The Derivative Formulas (`intercept_derivative` & `coef_derivative`)
Applying the chain rule to the loss function with respect to the intercept ($b$) and coefficients vector ($w$) yields the exact gradients used in the code loop:

* **Intercept Derivative:**
$$\frac{\partial J}{\partial b} = -2 \cdot (y_i - \hat{y}_i)$$

* **Coefficients Derivative:**
$$\frac{\partial J}{\partial w} = -2 \cdot (y_i - \hat{y}_i) \cdot X_i$$

---

### 2. The Parameter Update Formulas
The parameters are updated by subtracting a fraction of the gradient (controlled by the learning rate $\eta$) from the current values:

* **Intercept Update:**
$$b_{\text{new}} = b_{\text{old}} - \eta \cdot \left[ -2 \cdot (y_i - \hat{y}_i) \right]$$

* **Coefficients Update:**
$$w_{\text{new}} = w_{\text{old}} - \eta \cdot \left[ -2 \cdot (y_i - \hat{y}_i) \cdot X_i \right]$$


In [ ]:
from sklearn.datasets import load_diabetes

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

In [ ]:
X, y = load_diabetes(return_X_y = True)

In [ ]:
print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
reg = LinearRegression()
reg.fit(X_train, y_train)

LinearRegression()

In [ ]:
print(reg.coef_) # Beta_1ton
print(reg.intercept_) # Beta_0

[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]
151.88331005254167


In [ ]:
y_pred = reg.predict(X_test)
r2_score(y_test, y_pred)

0.4399338661568968

## Creating own regression class

In [11]:
class StochasticGradientDescent:

  def __init__(self, learning_rate, epochs):
    self.coef_ = None
    self.intercept_ = None
    self.lr = learning_rate
    self.epochs = epochs

  def fit(self, X_train, y_train):
    # initialize your coefficients
    self.intercept_ = 0
    self.coef_ = np.ones(X_train.shape[1])

    # training loop
    for i in range (self.epochs):

      # weights will be updated number of times there are rows
      for j in range (X_train.shape[0]):
        # select a random row index
        idx = np.random.randint(0, X_train.shape[0])
        y_hat = np.dot(X_train[idx], self.coef_) + self.intercept_ # prediction of idx row

        intercept_derivative = -2 * (y_train[idx] - y_hat)
        self.intercept_ = self.intercept_ - (self.lr * intercept_derivative)

        coef_derivative = -2 * np.dot((y_train[idx] - y_hat), X_train[idx])
        self.coef_ = self.coef_ - (self.lr * coef_derivative)

    print(self.intercept_, self.coef_)

  def predict(self, X_test):
    return np.dot(X_test, self.coef_) + self.intercept_


In [21]:
sgd = StochasticGradientDescent(epochs = 40, learning_rate = 0.01)

In [22]:
sgd.fit(X_train, y_train)

153.6782224078786 [  60.43369355  -56.64511359  312.29469806  221.32547842   33.74655267
   -1.16816967 -167.16518939  136.17537099  289.3460952   121.93681983]


In [23]:
y_pred = sgd.predict(X_test)

In [24]:
r2_score(y_test, y_pred)

0.4216165094266826

In [28]:
# Due to Randomness choosing index for updation, the results are not steady
result = []
for i in range (10):
  sgd = StochasticGradientDescent(epochs = 40, learning_rate = 0.01)
  sgd.fit(X_train, y_train)
  y_pred = sgd.predict(X_test)
  result.append(r2_score(y_test, y_pred))

168.2916458986797 [  59.0294657   -58.98149315  317.57160134  222.98975278   28.77347609
  -11.79314338 -160.5174851   127.07852533  296.04454164  127.80263922]
154.34136156231244 [  68.96814491  -44.58122284  316.58168829  223.55331817   30.37841258
  -10.51555118 -161.9824061   128.54160047  290.49846496  123.30155081]
153.84453284521837 [  57.77280703  -48.61735163  321.14130892  225.66692081   34.79087188
   -3.89341386 -167.90767404  138.42688384  296.45151499  123.79260979]
159.48446044385645 [  59.69857855  -50.24123854  314.67031735  230.95352055   29.91559662
   -8.31126708 -168.60554672  133.82090708  294.88206607  120.50458711]
151.372255117914 [  60.3151131   -56.07921201  319.69600776  221.7789643    22.59123027
  -20.15198059 -164.17311769  128.04323481  299.06391628  120.71077687]
143.90522224227618 [  60.34038412  -41.22444658  319.27221282  225.27084827   26.11666195
  -14.78322104 -162.7149486   126.2699325   293.87591392  120.73461051]
157.7791499321748 [  61.2860837

In [29]:
result

[0.3792064118680418,
 0.41900869957974796,
 0.4217879194947013,
 0.4128370549525251,
 0.42483120718940504,
 0.40772125044314855,
 0.4183098476702728,
 0.4118207855984063,
 0.42230720569965297,
 0.4094121393600505]

## Using Sklearn's SGD

In [30]:
from sklearn.linear_model import SGDRegressor

In [32]:
reg = SGDRegressor(max_iter = 100, learning_rate = 'constant', eta0 = 0.01)

In [33]:
reg.fit(X_train, y_train)

SGDRegressor(learning_rate='constant', max_iter=100)

In [34]:
y_pred = reg.predict(X_test)

In [35]:
r2_score(y_test, y_pred)

0.4317028302161877